# GARUDA — Train Helmet Classifier + License Plate Detector

Self-contained notebook. No dependency on cloning the repo — just run top to bottom.

**Before running:** `Runtime -> Change runtime type -> T4 GPU`

Produces a `garuda_results.zip` at the end containing:
- `helmet_cnn.pt` (trained weights, state_dict) + `helmet_metrics.json` + training curve / confusion matrix plot
- `plate_yolo.pt` (fine-tuned YOLO11n) + `plate_metrics.json` (mAP50, mAP50-95, precision, recall)

Download that zip and hand it back — it gets dropped into `ml/models/weights/` and wired into the live pipeline.

## 1. Install dependencies

In [ ]:
!pip install -q ultralytics kaggle
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 2. Kaggle authentication
Kaggle now issues a single **API token** (starts with `KGAT_`) instead of the old `kaggle.json` file.

Go to **kaggle.com -> Settings -> API -> Create New Token**, copy the token string, then run the cell below.
It opens a **hidden** input box (masked, like a password field) -- the token is written straight to
`~/.kaggle/access_token` and is never saved inside this notebook file.


In [ ]:
import os
from getpass import getpass

kaggle_token = getpass('Paste your Kaggle API token (KGAT_...): ').strip()

os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/access_token', 'w') as f:
    f.write(kaggle_token)
os.chmod('/root/.kaggle/access_token', 0o600)

del kaggle_token  # don't keep the token in a notebook variable longer than needed
print('Kaggle token installed at ~/.kaggle/access_token')

## 3. Download datasets
- Helmet detection (With Helmet / Without Helmet, VOC bboxes): `andrewmvd/helmet-detection`
- Car license plate detection (VOC bbox, class "licence"): `andrewmvd/car-plate-detection`

If a slug 404s, search the dataset name on kaggle.com and update the `!kaggle datasets download -d <slug>` line.

In [ ]:
!mkdir -p data/raw/helmet data/raw/plate
!kaggle datasets download -d andrewmvd/helmet-detection -p data/raw/helmet --unzip
!kaggle datasets download -d andrewmvd/car-plate-detection -p data/raw/plate --unzip
!echo '--- helmet ---' && ls data/raw/helmet
!echo '--- plate ---' && ls data/raw/plate

## 4. Pascal VOC parsing utilities (shared by both data-prep steps)

In [ ]:
import xml.etree.ElementTree as ET
from dataclasses import dataclass
from pathlib import Path
from typing import List, Optional

@dataclass
class VocObject:
    name: str
    xmin: int; ymin: int; xmax: int; ymax: int

@dataclass
class VocAnnotation:
    image_path: Path
    width: int; height: int
    objects: List[VocObject]

def parse_voc_xml(xml_path: Path, images_dir: Path) -> Optional[VocAnnotation]:
    tree = ET.parse(xml_path); root = tree.getroot()
    filename = root.findtext('filename')
    image_path = images_dir / filename if filename else None
    if image_path is None or not image_path.exists():
        for ext in ('.png', '.jpg', '.jpeg'):
            cand = images_dir / f'{xml_path.stem}{ext}'
            if cand.exists():
                image_path = cand; break
    if image_path is None or not image_path.exists():
        return None
    size = root.find('size')
    width = int(size.findtext('width')) if size is not None else 0
    height = int(size.findtext('height')) if size is not None else 0
    objects = []
    for obj in root.findall('object'):
        name = obj.findtext('name', '').strip()
        bnd = obj.find('bndbox')
        if bnd is None: continue
        objects.append(VocObject(
            name,
            int(float(bnd.findtext('xmin'))), int(float(bnd.findtext('ymin'))),
            int(float(bnd.findtext('xmax'))), int(float(bnd.findtext('ymax'))),
        ))
    return VocAnnotation(image_path, width, height, objects)

def load_voc_dataset(root_dir: str) -> List[VocAnnotation]:
    root = Path(root_dir)
    images_dir = root / 'images'
    ann_dir = root / 'annotations'
    if not ann_dir.exists(): ann_dir = root / 'annotation'
    if not images_dir.exists(): images_dir = root
    out = []
    for xml_path in sorted(ann_dir.glob('*.xml')):
        parsed = parse_voc_xml(xml_path, images_dir)
        if parsed is not None and parsed.objects:
            out.append(parsed)
    return out

def class_histogram(annotations):
    hist = {}
    for ann in annotations:
        for obj in ann.objects:
            hist[obj.name] = hist.get(obj.name, 0) + 1
    return hist

print('VOC utils loaded.')

## 5. Build helmet classification dataset (crops -> train/val/test/{helmet,no_helmet})

In [ ]:
import cv2, random

CROP_SIZE = 64
PADDING_RATIO = 0.15

def bucket(class_name):
    n = class_name.lower()
    if 'without' in n or 'no_helmet' in n or 'no-helmet' in n: return 'no_helmet'
    if 'helmet' in n: return 'helmet'
    return None

def build_helmet_dataset(src_dir, out_dir, train=0.70, val=0.15, seed=42):
    annotations = load_voc_dataset(src_dir)
    print('Raw class histogram:', class_histogram(annotations))
    out = Path(out_dir)
    for split in ('train', 'val', 'test'):
        for cls in ('helmet', 'no_helmet'):
            (out / split / cls).mkdir(parents=True, exist_ok=True)
    random.seed(seed); random.shuffle(annotations)
    n = len(annotations); n_train = int(n*train); n_val = int(n*val)
    split_map = {}
    for i, ann in enumerate(annotations):
        split_map[ann.image_path] = 'train' if i < n_train else ('val' if i < n_train+n_val else 'test')
    counts = {s: {'helmet': 0, 'no_helmet': 0} for s in ('train','val','test')}
    for ann in annotations:
        image = cv2.imread(str(ann.image_path))
        if image is None: continue
        h, w = image.shape[:2]
        split = split_map[ann.image_path]
        for idx, obj in enumerate(ann.objects):
            b = bucket(obj.name)
            if b is None: continue
            bw, bh = obj.xmax-obj.xmin, obj.ymax-obj.ymin
            px, py = int(bw*PADDING_RATIO), int(bh*PADDING_RATIO)
            x1, y1 = max(0, obj.xmin-px), max(0, obj.ymin-py)
            x2, y2 = min(w, obj.xmax+px), min(h, obj.ymax+py)
            crop = image[y1:y2, x1:x2]
            if crop.size == 0 or crop.shape[0] < 8 or crop.shape[1] < 8: continue
            crop = cv2.resize(crop, (CROP_SIZE, CROP_SIZE), interpolation=cv2.INTER_AREA)
            cv2.imwrite(str(out/split/b/f'{ann.image_path.stem}_{idx}.jpg'), crop, [cv2.IMWRITE_JPEG_QUALITY, 95])
            counts[split][b] += 1
    return counts

helmet_counts = build_helmet_dataset('data/raw/helmet', 'datasets/helmet')
print(helmet_counts)

## 6. Build plate detection YOLO dataset

In [ ]:
import shutil as _shutil

def build_plate_dataset(src_dir, out_dir, train=0.80, val=0.10, seed=42):
    annotations = load_voc_dataset(src_dir)
    out = Path(out_dir)
    for split in ('train', 'val', 'test'):
        (out/'images'/split).mkdir(parents=True, exist_ok=True)
        (out/'labels'/split).mkdir(parents=True, exist_ok=True)
    random.seed(seed); random.shuffle(annotations)
    n = len(annotations); n_train = int(n*train); n_val = int(n*val)
    counts = {'train':0,'val':0,'test':0}
    for i, ann in enumerate(annotations):
        split = 'train' if i < n_train else ('val' if i < n_train+n_val else 'test')
        dest_img = out/'images'/split/ann.image_path.name
        _shutil.copy2(ann.image_path, dest_img)
        lines = []
        for obj in ann.objects:
            cx = ((obj.xmin+obj.xmax)/2)/ann.width
            cy = ((obj.ymin+obj.ymax)/2)/ann.height
            bw = (obj.xmax-obj.xmin)/ann.width
            bh = (obj.ymax-obj.ymin)/ann.height
            lines.append(f'0 {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}')
        (out/'labels'/split/f'{dest_img.stem}.txt').write_text('\n'.join(lines))
        counts[split] += 1
    (out/'data.yaml').write_text(
        f'path: {out.resolve()}\ntrain: images/train\nval: images/val\ntest: images/test\n\n'
        f'nc: 1\nnames: ["license_plate"]\n'
    )
    return counts

plate_counts = build_plate_dataset('data/raw/plate', 'datasets/plate')
print(plate_counts)

## 7. HelmetCNN architecture
Must match `ml/models/helmet_cnn.py` exactly so the saved `state_dict` loads back into the repo unchanged.

In [ ]:
import torch.nn as nn

class SE(nn.Module):
    def __init__(self, channels, reduction=4):
        super().__init__()
        self.squeeze = nn.AdaptiveAvgPool2d(1)
        self.excitate = nn.Sequential(
            nn.Flatten(), nn.Linear(channels, channels//reduction), nn.Hardswish(inplace=True),
            nn.Linear(channels//reduction, channels), nn.Hardsigmoid(inplace=True),
        )
    def forward(self, x):
        s = self.excitate(self.squeeze(x))
        return x * s.unsqueeze(-1).unsqueeze(-1)

class HelmetCNN(nn.Module):
    CLASSES = ['no_helmet', 'helmet']
    INPUT_SIZE = 64
    def __init__(self, dropout=0.3, pretrained_mobilenet=True):
        super().__init__()
        if pretrained_mobilenet:
            self._build_from_mobilenet(dropout)
        else:
            raise NotImplementedError('use pretrained_mobilenet=True in this notebook')
    def _build_from_mobilenet(self, dropout):
        from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights
        backbone = mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.IMAGENET1K_V1)
        self.features = backbone.features
        self.avgpool = backbone.avgpool
        self.classifier = nn.Sequential(
            nn.Linear(576, 256), nn.Hardswish(inplace=True), nn.Dropout(dropout), nn.Linear(256, 2),
        )
        self._mode = 'mobilenet'
    def forward(self, x):
        x = self.features(x); x = self.avgpool(x); x = torch.flatten(x, 1)
        return self.classifier(x)
    def count_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

print('HelmetCNN defined.')

## 8. Train HelmetCNN

In [ ]:
import numpy as np, json
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
MEAN, STD = [0.485,0.456,0.406], [0.229,0.224,0.225]

train_tf = transforms.Compose([
    transforms.Resize((64,64)), transforms.RandomHorizontalFlip(0.5),
    transforms.ColorJitter(0.4,0.4,0.3), transforms.RandomRotation(10),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD),
])
eval_tf = transforms.Compose([
    transforms.Resize((64,64)), transforms.ToTensor(), transforms.Normalize(MEAN, STD),
])

root = Path('datasets/helmet')
train_ds = datasets.ImageFolder(root/'train', transform=train_tf)
val_ds   = datasets.ImageFolder(root/'val', transform=eval_tf)
test_ds  = datasets.ImageFolder(root/'test', transform=eval_tf)
print('Classes:', train_ds.class_to_idx)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=32, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=32, shuffle=False)

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    preds, labels = [], []
    for x, y in loader:
        x = x.to(device)
        p = model(x).argmax(1).cpu().numpy()
        preds.extend(p.tolist()); labels.extend(y.numpy().tolist())
    preds, labels = np.array(preds), np.array(labels)
    tp = int(np.sum((preds==1)&(labels==1))); tn = int(np.sum((preds==0)&(labels==0)))
    fp = int(np.sum((preds==1)&(labels==0))); fn = int(np.sum((preds==0)&(labels==1)))
    acc = (tp+tn)/max(1,len(labels)); prec = tp/max(1,tp+fp); rec = tp/max(1,tp+fn)
    f1 = 2*prec*rec/max(1e-9, prec+rec)
    return {'accuracy':round(acc,4), 'precision':round(prec,4), 'recall':round(rec,4),
            'f1':round(f1,4), 'confusion_matrix':{'tp':tp,'tn':tn,'fp':fp,'fn':fn}, 'n_samples':len(labels)}

model = HelmetCNN(dropout=0.3, pretrained_mobilenet=True).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
EPOCHS = 25
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
criterion = nn.CrossEntropyLoss()

os.makedirs('models/weights', exist_ok=True)
history = {'train_loss':[], 'val_accuracy':[], 'val_f1':[]}
best_f1 = -1.0

for epoch in range(1, EPOCHS+1):
    model.train(); running = 0.0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward(); optimizer.step()
        running += loss.item() * x.size(0)
    scheduler.step()
    train_loss = running / len(train_loader.dataset)
    val_metrics = evaluate(model, val_loader)
    history['train_loss'].append(round(train_loss,4))
    history['val_accuracy'].append(val_metrics['accuracy'])
    history['val_f1'].append(val_metrics['f1'])
    print(f"Epoch {epoch:2d}/{EPOCHS} | loss={train_loss:.4f} | val_acc={val_metrics['accuracy']:.4f} | val_f1={val_metrics['f1']:.4f}")
    if val_metrics['f1'] > best_f1:
        best_f1 = val_metrics['f1']
        torch.save(model.state_dict(), 'models/weights/helmet_cnn.pt')
        print(f'  -> new best, saved (val_f1={best_f1:.4f})')

model.load_state_dict(torch.load('models/weights/helmet_cnn.pt', map_location=device))
final_metrics = evaluate(model, test_loader)
final_metrics['class_to_idx'] = train_ds.class_to_idx
final_metrics['history'] = history
final_metrics['params'] = model.count_params()
with open('models/weights/helmet_metrics.json','w') as f:
    json.dump(final_metrics, f, indent=2)
print('FINAL TEST METRICS:', {k:v for k,v in final_metrics.items() if k not in ('history','class_to_idx')})

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11,4.5))
axes[0].plot(history['train_loss'], label='train_loss')
axes[0].plot(history['val_accuracy'], label='val_accuracy')
axes[0].plot(history['val_f1'], label='val_f1')
axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].set_title('Training Curves')

cm = final_metrics['confusion_matrix']
matrix = np.array([[cm['tn'], cm['fp']],[cm['fn'], cm['tp']]])
axes[1].imshow(matrix, cmap='Blues')
axes[1].set_xticks([0,1], ['no_helmet','helmet']); axes[1].set_yticks([0,1], ['no_helmet','helmet'])
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('Actual')
axes[1].set_title(f"Confusion Matrix (acc={final_metrics['accuracy']:.3f})")
for i in range(2):
    for j in range(2):
        axes[1].text(j, i, str(matrix[i,j]), ha='center', va='center')
fig.tight_layout()
fig.savefig('models/weights/helmet_training_report.png', dpi=130)
plt.show()

## 9. Train license plate YOLO detector

In [ ]:
import shutil

from ultralytics import YOLO

yolo_device = 0 if torch.cuda.is_available() else 'cpu'
plate_model = YOLO('yolo11n.pt')
plate_results = plate_model.train(
    data='datasets/plate/data.yaml', epochs=60, imgsz=640, batch=16,
    device=yolo_device, project='runs/detect', name='garuda_plate', patience=15, exist_ok=True,
)
plate_metrics = plate_model.val(data='datasets/plate/data.yaml', device=yolo_device)

best_ckpt = Path(plate_results.save_dir) / 'weights' / 'best.pt'
shutil.copy2(best_ckpt, 'models/weights/plate_yolo.pt')

plate_summary = {
    'mAP50': round(float(plate_metrics.box.map50), 4),
    'mAP50_95': round(float(plate_metrics.box.map), 4),
    'precision': round(float(plate_metrics.box.mp), 4),
    'recall': round(float(plate_metrics.box.mr), 4),
    'epochs': 60, 'imgsz': 640,
}
with open('models/weights/plate_metrics.json', 'w') as f:
    json.dump(plate_summary, f, indent=2)
print('PLATE DETECTOR METRICS:', plate_summary)

## 10. Package results and download

In [ ]:
import glob
import shutil


# Pull in the YOLO run's own confusion matrix / PR curve plots too
for f in glob.glob(f"{plate_results.save_dir}/*.png"):
    shutil.copy2(f, 'models/weights/')

shutil.make_archive('garuda_results', 'zip', 'models/weights')
print('Created garuda_results.zip with:')
!ls -la models/weights

from google.colab import files as _files
_files.download('garuda_results.zip')